### Importing libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

### Importing data

In [2]:
currency_index= pd.read_csv("../data/currency_data_processed/currency_index.csv", sep=";", low_memory=False)
players = pd.read_csv("../data/players_data_processed/players_processed.csv", sep=";", low_memory=False)

# Players activity ecnomics

### Adding **Day** Column

In [3]:
#first day of each league
players["Date"] = pd.to_datetime(players["Date"])
league_start = players.groupby("League")["Date"].transform("min")

# league day
players["Day"] = (players["Date"]-league_start).dt.days + 1
players_day=players.drop(columns=["Date"]).copy()

#exporting players dataset with Day column
players_day.reset_index(drop=True).to_csv("../data/players_data_processed/players_processed_day.csv",sep=";",index=False)

### Merging datasets

In [4]:
economy_population = currency_index.merge(players_day,on=["League", "Day"],how="left")
economy_population.head()

,League,Day,currency_index,Players
0,Affliction,1,1.000000,144193.0
1,Affliction,2,0.687330,144326.0
2,Affliction,3,0.954545,124599.0
3,Affliction,4,1.128216,116564.0
4,Affliction,5,1.357263,109595.0


### Players number normalisation
As the number of players differ from leagues

In [5]:
# player population on day 1 
day1_players = (economy_population.sort_values(["League", "Day"]).groupby("League", as_index=False).first()[["League", "Players"]].rename(columns={"Players": "Players_day1"}))

# Merge population with day 1 players
economy_population = economy_population.merge(day1_players,on="League",how="left")

# Number of players normalisation
economy_population["Players_norm"] = (economy_population["Players"] /economy_population["Players_day1"])
economy_population.head()

,League,Day,currency_index,Players,Players_day1,Players_norm
0,Affliction,1,1.000000,144193.0,144193.0,1.000000
1,Affliction,2,0.687330,144326.0,144193.0,1.000922
2,Affliction,3,0.954545,124599.0,144193.0,0.864113
3,Affliction,4,1.128216,116564.0,144193.0,0.808389
4,Affliction,5,1.357263,109595.0,144193.0,0.760058


### Correlation between number of players and Currency Purchasing Index 

#### Currency index vs Players plot

In [6]:
fig = go.Figure()

for league, df_league in economy_population.groupby("League"):
    
    # currency index
    fig.add_trace(go.Scatter(x=df_league["Day"],y=df_league["currency_index"],name=f"{league} — currency index",yaxis="y1"))
    # player population
    fig.add_trace(go.Scatter(x=df_league["Day"],y=df_league["Players"],name=f"{league} — Players",yaxis="y2",line=dict(dash="dash")))

fig.update_layout(title="Currency index and player population in time",xaxis_title="Day of league",yaxis=dict(title="Currency index"),yaxis2=dict(title="Player population",overlaying="y",side="right"))
fig.show()

In the plot it is visible that for the **Ancestor league** a strong negative correlation appears: as the number of players drops, the currency index increases. In the case of the other leagues the correlation is not very clear, however there are some periods that strongly indicate it. For example, in the **Settlers league** around day 210 a visible spike in the number of players corresponds with a rapid decrease in the currency index. This might lead to the conclusion that at the late stage of the game in a stable league a change in the currency index is only correlated with player activity.

#### Correlation calculations

Pearson correlation measures the strength of a linear relationship between two variables. In this context, it indicates whether changes in player population are proportionally associated with changes in the currency inflation index. Also Spearman correlation was used as it  measures a monotonic relationship based on the ranking of values, it is more resistant to outliers, non-linear relationships, and short-term fluctuations in the data.

Using both measures increases the reliability of the analysis. If Pearson and Spearman produce similar results, it suggests that the observed relationship is consistent and not driven by extreme observations or non-linear behaviour.

In [7]:
# Pearson correlation 
pearson_corr = (economy_population.groupby("League")[["Players_norm", "currency_index"]].corr(method="pearson").unstack().iloc[:,1].reset_index(name="pearson_corr"))

# Spearman correlation 
spearman_corr = (economy_population.groupby("League")[["Players_norm", "currency_index"]].corr(method="spearman").unstack().iloc[:,1].reset_index(name="spearman_corr"))

# Merging results
correlation_summary = pearson_corr.merge(spearman_corr,on="League")
correlation_summary

,League,pearson_corr,spearman_corr
0,Affliction,-0.607559,-0.510147
1,Ancestor,-0.926210,-0.970830
2,Keepers,-0.542178,-0.426656
3,Mercenaries,-0.361334,-0.018029
4,Necropolis,-0.100340,0.405536
5,Settlers,-0.669447,-0.744255


In [8]:
fig = px.scatter( economy_population,x="Players_norm", y="currency_index",color="League",title="Player population vs currency inflation",)
fig.update_layout(xaxis_title="Normalised player population",yaxis_title="Currency purchasing power index")
fig.show()

* The results show that most leagues exhibit a negative correlation between player population and currency inflation. This means that as the number of active players declines, the currency index increases. This pattern suggests that when player population decreases, the supply of items and crafting resources likely falls faster than demand, leading to rising prices and inflation of the Chaos Orb.

* The strength of this relationship varies between leagues:

    -**Ancestor league** shows a very strong negative correlation (Pearson −0.93, Spearman −0.97), indicating that in this case inflation was closely linked to the decline in player population and that number of players was the main factor controlling the currency index. 

    -**Settlers, Affliction and Keepers leagues** show moderate negative correlation, suggesting that the league economy was sensitive to population changes, however, with weaker effects.

    -Analysis of all the correlations calculated for **Mercenaries and Necropolis leagues** indicate that there is no relationship between population and inflation, suggesting a more stable or structurally different market.

* Across most leagues, the analysis indicates that declining player population is associated with increasing inflation, meaning that the purchasing power of Chaos Orb tends to weaken as fewer players remain active in the league economy. 

### Short term market dynamics

In [9]:
economy_population = economy_population.sort_values(["League", "Day"]).copy()

# Daily change of currency index 
economy_population["daily_change_index"] = (economy_population.groupby("League")["currency_index"].pct_change())

# Daily change of player population                                               ]
economy_population["daily_change_players"] = (economy_population.groupby("League")["Players_norm"].pct_change())
economy_population[["daily_change_index","daily_change_players"]] = (economy_population[["daily_change_index","daily_change_players"]].fillna(0))

# Limiting the scope of population change to avoid outliers
economy_population_filtered = economy_population[economy_population["daily_change_players"].between(-0.2, 0.2)].copy()

#exporting dataset
economy_population.reset_index(drop=True).to_csv("../data/players_data_processed/population_economy.csv",sep=";",index=False)

economy_population.head()

C:\Users\miood\AppData\Local\Temp\ipykernel_13976\206507701.py:7: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  economy_population["daily_change_players"] = (economy_population.groupby("League")["Players_norm"].pct_change())


,League,Day,currency_index,Players,Players_day1,Players_norm,daily_change_index,daily_change_players
0,Affliction,1,1.000000,144193.0,144193.0,1.000000,0.000000,0.000000
1,Affliction,2,0.687330,144326.0,144193.0,1.000922,-0.312670,0.000922
2,Affliction,3,0.954545,124599.0,144193.0,0.864113,0.388773,-0.136684
3,Affliction,4,1.128216,116564.0,144193.0,0.808389,0.181940,-0.064487
4,Affliction,5,1.357263,109595.0,144193.0,0.760058,0.203017,-0.059787


In [10]:
fig = px.scatter( economy_population_filtered,x="daily_change_players",y="daily_change_index",color="League",title="Daily player change vs daily inflation")
fig.update_layout(xaxis_title="Daily change in normalised players population",yaxis_title="Daily change in currency index")
fig.show()

Short-term changes in player numbers rarely result in immediate price changes. The market tends to react slowly, rather than overnight.

Occasional observations appear in the upper-left part of the chart, where players population decreases while prices increase. This suggests that when players leave the league, the supply of items and currency on the market may drop, which can lead to higher prices. However, this effect appears only occasionally and is not a consistent daily pattern.

### League phases economy dynamics

#### Leagues segmentation

In [11]:
def league_phase(day):
    
    if day <= 7:
        return "Launch hype"
    elif day <= 30:
        return "Early league"
    elif day <= 70:
        return "Mid league"
    else:
        return "Late league"

economy_population["Phase"] = economy_population["Day"].apply(league_phase)

In [12]:
league_phase_summary = ( economy_population.groupby(["League","Phase"]).agg(avg_players=("Players_norm","mean"),avg_inflation=("currency_index","mean"),price_volatility=("daily_change_index","std")).reset_index())

#exporting dataset
league_phase_summary.reset_index(drop=True).to_csv("../data/players_data_processed/league_phase_summary.csv",sep=";",index=False)
league_phase_summary

,League,Phase,avg_players,avg_inflation,price_volatility
0,Affliction,Early league,0.637289,1.843685,0.021896
1,Affliction,Late league,0.103166,2.108470,0.017320
2,Affliction,Launch hype,0.844593,1.153814,0.217277
3,Affliction,Mid league,0.324601,1.993028,0.016999
4,Ancestor,Early league,0.611504,2.118366,0.023838
5,Ancestor,Late league,0.136457,3.386499,0.017821
6,Ancestor,Launch hype,0.865148,1.328447,0.081552
7,Ancestor,Mid league,0.299853,2.518866,0.016031
8,Keepers,Early league,0.472368,1.311909,0.049064
9,Keepers,Late league,0.089670,1.567722,0.142986


In [13]:
phase_order = ["Launch hype","Early league","Mid league","Late league"]

In [14]:
#population across league phases
fig = px.bar(league_phase_summary,x="Phase",y="avg_players",color="League",category_orders={"Phase": phase_order},title="Player population across league phases")
fig.update_layout(xaxis_title="League phase",yaxis_title="Average normalised player population")
fig.show()

In [15]:
heatmap_data = (league_phase_summary.pivot(index="League", columns="Phase", values="avg_players").reindex(columns=phase_order))
fig = px.imshow(heatmap_data,aspect="auto",color_continuous_scale="Blues",title="Player population across league phase")
fig.update_layout(xaxis_title="League phase",yaxis_title="League")
fig.show()

In [16]:
# Inflation across league phases
fig = px.bar(league_phase_summary,x="Phase",y="avg_inflation",color="League",category_orders={"Phase": phase_order},title="Currency inflation across league phases")
fig.update_layout(xaxis_title="League phase",yaxis_title="Average currency index")
fig.show()

In [17]:
heatmap_data = (league_phase_summary.pivot(index="League", columns="Phase", values="avg_inflation").reindex(columns=phase_order))
fig = px.imshow(heatmap_data,aspect="auto",color_continuous_scale="Blues",title="Currency inflation across league phases")
fig.update_layout(xaxis_title="League phase",yaxis_title="League")
fig.show()

In [18]:
# Volatility across league phases
fig = px.bar(league_phase_summary,x="Phase",y="price_volatility",color="League",category_orders={"Phase": phase_order},title="Price volatility across league phases")
fig.update_layout(xaxis_title="League phase",yaxis_title="Price volatility")
fig.show()

In [19]:
heatmap_data = (league_phase_summary.pivot(index="League", columns="Phase", values="price_volatility").reindex(columns=phase_order))
fig = px.imshow(heatmap_data,aspect="auto",color_continuous_scale="Blues",title="Price volatility across league phases")
fig.update_layout(xaxis_title="League phase",yaxis_title="League")
fig.show()

Analysing the plots presented above, several general conclusions can be made, with a few exceptions observed in particular leagues.:
1. **Launch hype phase**: biggest number of players, high volatility, low inflation
2. **Early-phase league**: population of players begin to decrease, market stabilises
3. **Mid-phase league**: population of players continously decreases, inflation rises
4. **Late-phase league**:  small number of players, stable market, the highest inflation

**Conclusions:**

1. Players population consistently declines in time, with the largest drop occurring shortly after the launch phase.
2. In most leagues there is a negative relationship between player population and inflation: when the number of active players decreases, the currency index tends to increase.
3. The influence of population on prices appears to be gradual rather than immediate. Daily changes in player numbers rarely lead to instant price changes. 
4. The Path of Exile league economy follows a clear lifecycle. Player population declines steadily over time, price volatility is highest during the launch phase, and inflation tends to increase as the league progresses.
5. Price volatility is highest at league launch, when prices are still forming and market activity is unstable.
6. Inflation tends to increase as the league progresses, probably due to declining player population and reduced market supply.

Player activity is an important driver of economic dynamics in Path of Exile leagues, affecting both inflation trends and market stability.
